# 01 · Baseline — M0 backtested with the competition's own score

## Executive summary (read this first)

**The question.** How good is a model on each unit, measured the way the leaderboard measures it? The leaderboard gives
every unit one number: each part of the score (marginal CRPS, joint variogram, tail) **divided by the same part of
M0's score** on that unit, then weighted by the card's weights. **M0 = 1.0; below 1 beats M0.** The final score is
the equal-weight mean over units.

**What this notebook does.**

1. **Backtest dates.** For every unit, the past dates inside its own history where the answer is already known (the
   value `horizon` panel rows later; for a log-return unit, the sum of the next `horizon` returns, exactly as
   `build_realized.py` defines the real answer). The **latest 30 %** of those dates is the backtest block; every 5th
   date is used (neighbouring dates share most of their answer window, so the others add little).
2. **M0 at every date**, rebuilt with the organizers' published recipe (`docs/M0-BASELINE.md`, reusing
   `m0_baseline.py`). Checked: at each unit's real as-of it reproduces `m0_baseline.draws` exactly, on all units.
3. **Score M0 at every date** with the official scorer: each category (marginal CRPS, joint variogram, tail) and their
   weighted sum. Per unit (§5) and per family: M0's mean score in each category over its backtest dates. These are the
   denominators every later model is divided by. Saved in `data/`.

**How a later model will be scored** (not done here): per unit, `Σ weight_k × mean(category_k of the model) ÷
mean(category_k of M0)` over the same backtest dates, a ratio of means. That is the leaderboard formula with a mean over
many dates in place of one date. It is steadier than averaging one ratio per date, because a single date where M0
happens to be very good would blow up that date's ratio.

**Why this design** (and the suggestions behind it):

- Each unit's own history is long (years of rows) while the leaderboard sees one date per unit, so a backtest over
  hundreds of past dates per unit measures a model far more precisely than the leaderboard can.
- M0 has nothing to tune: one fixed recipe for every unit, estimated at each date from that date's own past rows. So
  M0's backtest score cannot be overfit. A model that does have settings should be tuned on the first 70 % of each
  unit and judged on this last 30 %.
- The text half is off: there is no news corpus for past dates. This backtest measures the time-series part only.
- The real check stays the team eval's real outcomes (`run_eval.py`) and the submission itself.

No realized outcome is read: every answer is a later row of the unit's own history.

## 0 — Setup

In [1]:
import bisect, json, multiprocessing as mp, pathlib, subprocess, sys, tempfile, time, tomllib, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, OUT = HERE.parent, HERE.parent / "units", HERE / "data"
OUT.mkdir(exist_ok=True)
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S

pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
UNIT_IDS = sorted(p.name for p in UNITS.glob("t2-F*") if (p / "card.toml").exists())
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))
BACKTEST_FRAC, STRIDE, MIN_HISTORY = 0.30, 5, 100      # latest 30 % of dates, every 5th, >= 100 rows of history
print(f"{len(UNIT_IDS)} units; {WORKERS} workers")

103 units; 10 workers


## 1 — The competition score

From `m0_baseline.py` (the team's rebuild of the leaderboard formula, which reproduced 22 units of submission 01 to four
decimals): for one unit,

```
score = w_marginal × CRPS ÷ CRPS_M0  +  w_joint × variogram ÷ variogram_M0  +  w_tail × tail ÷ tail_M0
```

with the card's own weights (0.5 / 0.3 / 0.2; on a one-cell unit the joint weight is spread over the other two). M0
scores exactly 1.0 by construction. The leaderboard's number is the plain mean of this over units.

In [2]:
def parts(card, samples, y):
    """The scorer's three parts and their weighted composite, raw. samples (n_draws, n_cells), card order."""
    p = card.get("scoring", {}).get("params", {})
    w = m0.card_weights(card, samples.shape[1])
    return S._composite(samples, y, weights=w, tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))),
                        joint=S.card_joint_statistic(card), tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)),
                        ref_scale=None)

def competition_score(model, base, w):
    """Leaderboard formula. model / base: dicts (or Series) with marginal, joint, tail."""
    s = w[0] * model["marginal"] / base["marginal"] + w[2] * model["tail"] / base["tail"]
    return s + (w[1] * model["joint"] / base["joint"] if w[1] else 0.0)

print("weights, a one-cell card:", m0.card_weights({"scoring": {"params": {}}}, 1),
      "  a four-cell card:", m0.card_weights({"scoring": {"params": {}}}, 4))

weights, a one-cell card: (0.7142857142857143, 0.0, 0.28571428571428575)   a four-cell card: (0.5, 0.3, 0.2)


## 2 — The backtest dates and their answers

For each unit: each asset's history up to the card's as-of, read with M0's own rule (the first panel file holding the
asset). A **date** is a row present for every asset with at least 100 rows of history before it and its answer inside
the history. The answer for horizon `h` is the value `s` rows later (`s = h` on daily panels; on the four monthly units,
M0's own step count, e.g. 2 months for a 21-business-day card), or for a log-return unit the sum of the next `s` rows.
The latest 30 % of the dates is the backtest block; every 5th date of it is used.

In [3]:
def load_unit(u):
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof, ttype = card["task"]["id"], card["provenance"]["data_cutoff"], t.get("target_type", "level")
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):                     # M0 3.1: the first panel holding the asset
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    steps = {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": ttype, "asof": asof, "rows": rows, "steps": steps,
            "dates": {a: [d for d, _ in rows[a]] for a in assets}}

def backtest_dates(U):
    """-> list of (origin date, answers in card order: asset-major, then horizon)."""
    pos = {a: {d: i for i, d in enumerate(U["dates"][a])} for a in U["assets"]}
    common = sorted(set.intersection(*(set(U["dates"][a]) for a in U["assets"])))
    smax = max(U["steps"].values())
    ok = [d for d in common if all(MIN_HISTORY <= pos[a][d] <= len(U["rows"][a]) - 1 - smax for a in U["assets"])]
    block = ok[int(np.floor(len(ok) * (1 - BACKTEST_FRAC))):][::STRIDE]
    out = []
    for d in block:
        y = []
        for a in U["assets"]:
            i, vals = pos[a][d], U["rows"][a]
            for h in U["horizons"]:
                s = U["steps"][h]
                y.append(sum(v for _, v in vals[i + 1: i + s + 1]) if U["target_type"] == "log_return" else vals[i + s][1])
        out.append((d, np.array(y)))
    return out, len(ok)

t0 = time.time()
UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
DATES, rows = {}, []
for u, U in UNITS_DATA.items():
    DATES[u], n_ok = backtest_dates(U)
    rows.append({"unit": u, "family": U["family"], "assets": len(U["assets"]), "horizons": str(U["horizons"]),
                 "eligible dates": n_ok, "backtest dates": len(DATES[u]),
                 "first": DATES[u][0][0] if DATES[u] else None, "last": DATES[u][-1][0] if DATES[u] else None})
SPLIT = pd.DataFrame(rows).set_index("unit")
print(f"{time.time() - t0:.0f}s; {SPLIT['backtest dates'].sum():,} backtest dates over {len(SPLIT)} units")
display(SPLIT.groupby("family").agg(units=("assets", "size"), dates=("backtest dates", "sum"),
                                    min_per_unit=("backtest dates", "min"), median_per_unit=("backtest dates", "median"),
                                    first=("first", "min"), last=("last", "max")))
print("units with the fewest backtest dates:"); display(SPLIT.nsmallest(5, "backtest dates"))

3s; 23,635 backtest dates over 103 units


,units,dates,min_per_unit,median_per_unit,first,last
family,,,,,,
T2-F1,23,4956,11,249,2002-02-14,2024-03-15
T2-F2,27,5930,52,208,2000-05-02,2024-06-03
T2-F3,22,6004,64,300,2000-08-22,2024-07-31
T2-F4,31,6745,9,235,2005-05-06,2024-06-27


units with the fewest backtest dates:


,family,assets,horizons,eligible dates,backtest dates,first,last
unit,,,,,,,
t2-F4-covid-nfp-2020,T2-F4,1,[21],140,9,2016-07-01,2019-11-01
t2-F4-cpi-vintage-2022,T2-F4,1,[21],166,10,2018-01-01,2021-10-01
t2-F1-cpi-glidepath-2023,T2-F1,1,"[140, 160]",172,11,2018-05-01,2022-07-01
t2-F1-sahm-watch-2024,T2-F1,1,"[145, 165]",185,12,2019-02-01,2023-09-01
t2-F1-considerable-period-2003,T2-F1,1,"[126, 189]",614,37,2002-02-14,2002-11-01


## 3 — M0 at any date

The same code as `m0_baseline.draws` (its own `_steps`, gap rule, alignment, drift, covariance, jitter and seed), with
the as-of moved to the backtest date: the last 300 rows up to that date. Check: at each unit's real as-of it gives
exactly `m0_baseline.draws`.

In [4]:
def m0_draws(U, origin):
    """M0's 500 draws at `origin` (a date string), shape (500, n_assets, n_horizons), card order."""
    assets, horizons, tt = U["assets"], U["horizons"], U["target_type"]
    by_asset = sorted(assets)
    steps, anchor = {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        steps[a], last = m0._steps(U["rows"][a][max(0, k - m0._WINDOW): k], tt)
        anchor[a] = last if tt == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]
    s = U["steps"]
    mean = np.array([anchor[a] + s[h] * mu[ai[a]] for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10
    jittered = cov + 1e-9 * np.eye(len(cells))
    try:
        factor = np.linalg.cholesky(jittered)
    except np.linalg.LinAlgError:
        factor = np.diag(np.sqrt(np.diag(jittered)))
    rng = np.random.default_rng(zlib.crc32(U["id"].encode()) & 0x7FFFFFFF)
    samples = mean + rng.standard_normal((m0.N_DRAWS, len(cells))) @ factor.T
    out = np.empty((m0.N_DRAWS, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        out[:, assets.index(a), horizons.index(h)] = samples[:, ci]
    return out

diffs = {u: float(np.abs(m0_draws(U, U["asof"]) - m0.draws(UNITS / u)).max()) for u, U in UNITS_DATA.items()}
print(f"m0_draws at the real as-of vs m0_baseline.draws: max |difference| over {len(diffs)} units = {max(diffs.values()):.1e}")
assert max(diffs.values()) == 0

m0_draws at the real as-of vs m0_baseline.draws: max |difference| over 103 units = 0.0e+00


## 4 — Score M0 at every backtest date

At each date: M0's 500 draws against the answer, scored with the official scorer's categories. The check below compares
one date with `scoring/scoring.py` itself.

In [5]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

def score_unit(u):
    U = UNITS_DATA[u]
    return [{"unit": u, "family": U["family"], "origin": origin,
             **parts(U["card"], m0_draws(U, origin).reshape(m0.N_DRAWS, -1), y)} for origin, y in DATES[u]]

# the scorer check, one date of a two-asset, two-horizon unit
u = "t2-F3-inversion-persistence-2019"; U = UNITS_DATA[u]; origin, y = DATES[u][10]
d3 = m0_draws(U, origin)
with tempfile.TemporaryDirectory() as tmp:
    out_dir, real = pathlib.Path(tmp) / "out", pathlib.Path(tmp) / "realized.parquet"; out_dir.mkdir()
    cells = [(a, h) for a in U["assets"] for h in U["horizons"]]
    pd.DataFrame([{"draw": np.int32(d), "asset": a, "horizon": np.int32(h), "value": float(d3[d, ai, hi])}
                  for d in range(len(d3)) for ai, a in enumerate(U["assets"]) for hi, h in enumerate(U["horizons"])]
                 ).to_parquet(out_dir / "forecast.parquet", index=False)
    (out_dir / "forecast_meta.json").write_text(json.dumps({"unit_id": U["id"], "asof": U["asof"], "representation": "samples",
        "asset_ids": U["assets"], "horizons": U["horizons"], "n_draws": len(d3), "target": U["target_type"]}))
    (out_dir / "forecast_rationale.md").write_text("# check\n")
    pd.DataFrame([{"asset": a, "horizon": h, "value": float(v)} for (a, h), v in zip(cells, y)]).to_parquet(real, index=False)
    r = subprocess.run([sys.executable, str(REPO / "scoring" / "scoring.py"), "score", "--card", str(UNITS / u / "card.toml"),
                        "--forecast", str(out_dir / "forecast.parquet"), "--realized", str(real)], capture_output=True, text=True, cwd=REPO)
off, mine = json.loads(r.stdout), parts(U["card"], d3.reshape(len(d3), -1), y)
check = pd.DataFrame({"official": [off["marginal_crps"], off["joint_variogram"], off["tail_penalty"], off["composite_score"]],
                      "this notebook": [mine["marginal"], mine["joint"], mine["tail"], mine["composite"]]},
                     index=["marginal CRPS", "joint variogram", "tail", "composite"])
display(check); assert np.allclose(check["official"], check["this notebook"], rtol=0, atol=1e-12)

t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    SCORES = pd.DataFrame([r for rows in pool.map(score_unit, UNIT_IDS, chunksize=1) for r in rows])
print(f"scored M0 on {len(SCORES):,} dates in {time.time() - t0:.0f}s")

,official,this notebook
marginal CRPS,0.2157,0.2157
joint variogram,0.2165,0.2165
tail,0.01548,0.01548
composite,0.1759,0.1759


scored M0 on 23,635 dates in 1s


## 5 — M0's score per unit: each category and the weighted sum

For each unit, the mean over its backtest dates of each category, the card's weights, each weighted part, and the
weighted sum (`w_marginal × CRPS + w_joint × variogram + w_tail × tail`, the scorer's composite). Lower is better. The
scores are raw, in the unit's own units (a yield in percentage points, a return in log units), so compare units
through ratios, not raw numbers. On a one-cell unit the joint category does not exist and its weight is spread over the
other two.

In [6]:
cats = {"marginal": "CRPS (marginal)", "joint": "variogram (joint)", "tail": "tail"}
rows = []
for u in UNIT_IDS:
    U, w = UNITS_DATA[u], m0.card_weights(UNITS_DATA[u]["card"], len(UNITS_DATA[u]["assets"]) * len(UNITS_DATA[u]["horizons"]))
    m = SCORES[SCORES.unit == u][["marginal", "joint", "tail", "composite"]].mean()
    r = {"family": U["family"], "unit": u, "dates": int((SCORES.unit == u).sum()),
         **{cats[k]: m[k] for k in cats},
         "w CRPS": w[0], "w variogram": w[1], "w tail": w[2],
         "w x CRPS": w[0] * m["marginal"], "w x variogram": w[1] * m["joint"], "w x tail": w[2] * m["tail"]}
    r["weighted sum"] = r["w x CRPS"] + r["w x variogram"] + r["w x tail"]
    assert np.isclose(r["weighted sum"], m["composite"], rtol=1e-12, atol=1e-15), u    # = the scorer's composite
    rows.append(r)
M0_UNIT = pd.DataFrame(rows).set_index(["family", "unit"])
print("M0 on each unit's backtest block: mean score per category, weights, weighted parts and their sum (lower is better)")
M0_UNIT

M0 on each unit's backtest block: mean score per category, weights, weighted parts and their sum (lower is better)


dates  CRPS (marginal)  variogram (joint)      tail  w CRPS  w variogram  w tail  w x CRPS  w x variogram  w x tail  weighted sum
family unit                                                                                                                                                                
T2-F1  t2-F1-ai-mom-2024                    355          0.06966                  0  0.007328  0.7143            0  0.2857   0.04976              0  0.002094       0.05185
       t2-F1-aud-on-hold-2016               237          0.05387            0.01423   0.00597     0.5          0.3     0.2   0.02694       0.004269  0.001194        0.0324
       t2-F1-cad-boc-2017                   247          0.04505            0.01721  0.006385     0.5          0.3     0.2   0.02252       0.005162  0.001277       0.02896
       t2-F1-chf-highly-valued-2021         306          0.02717            0.01135  0.003899     0.5          0.3     0.2   0.01359       0.003406 0.0007798       0.01777
       t2-F1-conflicting-texts-2024         350           0.4825             0.1289   0.07003     0.5          0.3     0.2    0.2412        0.03868   0.01401        0.2939
       t2-F1-considerable-period-2003        37           0.5485             0.1378   0.05446     0.5          0.3     0.2    0.2743        0.04135   0.01089        0.3265
       t2-F1-conundrum-2005                  60           0.5009             0.1274   0.06143     0.5          0.3     0.2    0.2505        0.03823   0.01229         0.301
       t2-F1-cpi-glidepath-2023              11            4.767             0.3937     1.586     0.5          0.3     0.2     2.383         0.1181    0.3171         2.819
       t2-F1-dkk-peg-2019                   275           0.3274                  0   0.05177  0.7143            0  0.2857    0.2339              0   0.01479        0.2487
       t2-F1-eur-range-2017                 250          0.07121             0.0164   0.01056     0.5          0.3     0.2   0.03561       0.004919  0.002113       0.04264
       t2-F1-fed-put-2019                   280          0.04127                  0  0.005081  0.7143            0  0.2857   0.02948              0  0.001452       0.03093
       t2-F1-greater-confidence-2024        345           0.7425             0.2727    0.1726     0.5          0.3     0.2    0.3712        0.08182   0.03453        0.4876
       t2-F1-hawkish-cut-2024               358           0.7822             0.2735    0.1759     0.5          0.3     0.2    0.3911        0.08205   0.03517        0.5083
       t2-F1-jpy-ycc-flex-2018              263            7.942              1.577     1.313     0.5          0.3     0.2     3.971         0.4731    0.2627         4.707
       t2-F1-liftoff-telegraph-2015         221             0.11            0.06778   0.01584     0.5          0.3     0.2   0.05502        0.02033  0.003168       0.07852
       t2-F1-measured-pace-2004              51           0.6349             0.1664   0.06234     0.5          0.3     0.2    0.3174        0.04993   0.01247        0.3798
       t2-F1-normalization-2017             249           0.3744                  0   0.03464  0.7143            0  0.2857    0.2674              0  0.009898        0.2773
       t2-F1-patient-dropped-2015           215           0.1172                  0   0.01606  0.7143            0  0.2857   0.08369              0  0.004589       0.08828
       t2-F1-pause-2006                      82           0.3104              1.324   0.03875     0.5          0.3     0.2    0.1552         0.3971   0.00775        0.5601
       t2-F1-qe3-end-2014                   206           0.5525             0.1359   0.05187     0.5          0.3     0.2    0.2762        0.04077   0.01037        0.3274
       t2-F1-sahm-watch-2024                 12            1.814               1.64    0.5895     0.5          0.3     0.2    0.9072          0.492    0.1179         1.517
       t2-F1-third-cut-2019                 281           0.2717            0.04823   

**Per family.** The mean over the family's units of each category and of the weighted sum, with the median weighted sum
(the mean is led by the units with large raw scores, such as JPY in yen).

In [7]:
agg = {"units": ("dates", "size"), "dates": ("dates", "sum"), **{c: (c, "mean") for c in cats.values()},
       "weighted sum (mean)": ("weighted sum", "mean"), "weighted sum (median)": ("weighted sum", "median")}
M0_FAMILY = M0_UNIT.groupby("family").agg(**agg)
M0_FAMILY.loc["all"] = [len(M0_UNIT), M0_UNIT.dates.sum(), *[M0_UNIT[c].mean() for c in cats.values()],
                         M0_UNIT["weighted sum"].mean(), M0_UNIT["weighted sum"].median()]
M0_FAMILY[["units", "dates"]] = M0_FAMILY[["units", "dates"]].astype(int)
M0_FAMILY

,units,dates,CRPS (marginal),variogram (joint),tail,weighted sum (mean),weighted sum (median)
family,,,,,,,
T2-F1,23,4956,0.9121,0.2806,0.1925,0.5878,0.2773
T2-F2,27,5930,0.5591,0.002453,0.08229,0.421,0.1789
T2-F3,22,6004,0.426,2.004,0.06031,0.8264,0.7895
T2-F4,31,6745,4.509,0.0007707,0.5432,3.376,0.0362
all,103,23635,1.798,0.4917,0.2409,1.434,0.2024


## 6 — Save

In [8]:
rows = [{"unit": u, "origin": o, "asset": a, "horizon": h, "steps": UNITS_DATA[u]["steps"][h], "answer": float(v)}
        for u in UNIT_IDS for o, y in DATES[u]
        for (a, h), v in zip([(a, h) for a in UNITS_DATA[u]["assets"] for h in UNITS_DATA[u]["horizons"]], y)]
pd.DataFrame(rows).to_parquet(OUT / "backtest_answers.parquet", index=False)
SCORES.to_parquet(OUT / "m0_backtest_scores.parquet", index=False)
M0_UNIT.to_csv(OUT / "m0_per_unit.csv")
M0_FAMILY.to_csv(OUT / "m0_per_family.csv")
for f in sorted(p.name for p in OUT.iterdir()):
    print(f"data/{f:<28} {(OUT / f).stat().st_size / 1e3:8.1f} kB")

data/backtest_answers.parquet        328.1 kB
data/m0_backtest_scores.parquet      867.1 kB
data/m0_per_family.csv                 0.7 kB
data/m0_per_unit.csv                  19.4 kB


## 7 — In short

In [9]:
print(f"Backtest: the latest 30 % of each unit's history, every 5th date: {SPLIT['backtest dates'].sum():,} dates over {len(UNIT_IDS)} units.")
print("M0 is rebuilt exactly (difference 0 at every real as-of) and scored with the official scorer (equal to 12 decimals).")
print("M0's mean score per unit, by category and weighted sum, is in §5 and data/m0_per_unit.csv; per family in data/m0_per_family.csv.")
print("A later model's competition score on a unit = its mean category scores divided by these, weighted: M0 itself = 1.0.")

Backtest: the latest 30 % of each unit's history, every 5th date: 23,635 dates over 103 units.
M0 is rebuilt exactly (difference 0 at every real as-of) and scored with the official scorer (equal to 12 decimals).
M0's mean score per unit, by category and weighted sum, is in §5 and data/m0_per_unit.csv; per family in data/m0_per_family.csv.
A later model's competition score on a unit = its mean category scores divided by these, weighted: M0 itself = 1.0.
